# 05 Neural Network (PyTorch) - House Price Prediction

This is a complete, standalone pipeline for training a Deep Learning MLP model using PyTorch on dirty house price data.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import re

sns.set(style='whitegrid')

### 1. Data Cleaning & Normalization

In [ ]:
def clean_house_data(df):
    df = df.copy().dropna(subset=['Price']).drop_duplicates()
    def parse_lot(v):
        if pd.isna(v): return 10000
        v = str(v).lower()
        num = float(re.findall(r'\d+\.\d+|\d+', v)[0])
        if 'ac' in v: return num * 43560
        return num
    df['Lot_Size'] = df['Lot_Size'].apply(parse_lot)
    df['Bedrooms'] = df['Bedrooms'].apply(lambda x: sum([int(i) for i in str(x).split('+')]) if '+' in str(x) else int(re.sub(r'\D', '', str(x))))
    df['Price'] = df['Price'].clip(upper=df['Price'].quantile(0.95)) # Stronger clipping for NN stability
    return df.select_dtypes(include=[np.number])

df = clean_house_data(pd.read_csv('../_data/house_prices.csv'))
X = df.drop('Price', axis=1).fillna(0)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 2. PyTorch Model Definition

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

model = MLPModel(X_train.shape[1])
train_loader = DataLoader(HouseDataset(X_train_scaled, y_train), batch_size=32, shuffle=True)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 3. Training Loop (Complete)
Executing a 100-epoch training cycle.

In [ ]:
model.train()
for epoch in range(100):
    total_loss = 0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 20 == 0: print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.2f}")

### 4. Evaluation & Visuals

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()

print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Actual vs Predicted - PyTorch MLP')
plt.show()

### 5. Model Saving

In [ ]:
torch.save(model.state_dict(), '../_model/house_prediction_neural_network.pth')
print("PyTorch MLP Model saved.")